# Exercises XP: Vector Databases and RAG

## What you'll learn
- Vector search strategies (KNN, ANN) and evaluation.
- Vector database utility (similarity search, RAG).
- Differences between vector DBs, libraries, and plugins.
- Best practices for vector store usage and performance.
- How LMs use context; embedding generation and storage.
- Querying vector stores and applying LMs for QA with retrieved context.

## What you'll build
A functional RAG pipeline with FAISS and ChromaDB, plus QA over retrieved context using a Hugging Face model.

## 0. Setup
The course instructions pin old versions (`faiss-cpu==1.7.4`, `chromadb==0.3.21`, `numpy<2`, `pydantic<2`). Those wheels do not exist for recent Python versions (Colab now runs Python ≥ 3.11 with NumPy 2), so we install current releases instead. `faiss-cpu ≥ 1.8` ships its own OpenMP runtime, so `apt install libomp-dev` is no longer required, and recent FAISS/Chroma work with NumPy 2. We keep `transformers < 5` and `sentence-transformers < 6`, which are compatible with each other.

In [1]:
%pip install -q -U faiss-cpu chromadb "sentence-transformers<6" "transformers<5" accelerate

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
os.environ["USE_TF"] = "0"                    # transformers: use PyTorch only
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["ANONYMIZED_TELEMETRY"] = "False"

import json, time
from pathlib import Path
import numpy as np
import pandas as pd
import faiss
import torch
from sentence_transformers import SentenceTransformer, InputExample
import chromadb
from chromadb.config import Settings
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM, pipeline
from IPython.display import display
import chromadb as _c, sentence_transformers as _s, transformers as _t

os.makedirs("cache", exist_ok=True)
pd.set_option("display.max_colwidth", 110)
print("faiss", faiss.__version__, "| chromadb", _c.__version__, "| sentence-transformers", _s.__version__,
      "| transformers", _t.__version__, "| numpy", np.__version__)

faiss 1.15.1 | chromadb 1.5.9 | sentence-transformers 5.7.0 | transformers 4.57.6 | numpy 2.5.3


## 🌟 Exercise 1 · Data loading and preparation

In [3]:
# The CSV is ';'-separated and zipped; pandas reads the zip directly (local copy if present, otherwise the course repo)
LOCAL = Path("labelled_newscatcher_dataset.csv")
URL = ("https://media.githubusercontent.com/media/devtlv/Datasets-GEN-AI-Bootcamp/main/Week%205/"
       "Day%204%20-%20Diving%20Deep%20into%20Vector%20Databases%20and%20RAG%20Chatbots/labelled_newscatcher_dataset.zip")
data_path = LOCAL if LOCAL.exists() else URL
pdf = pd.read_csv(data_path, sep=";")

# Add a unique identifier column
if "id" not in pdf.columns:
    pdf["id"] = range(len(pdf))

display(pdf)

,topic,link,domain,published_date,title,lang,id
0,SCIENCE,https://www.eurekalert.org/pub_releases/2020-08/dbnl-acl080620.php,eurekalert.org,2020-08-06 13:59:45,A closer look at water-splitting's solar fuel potential,en,0
1,SCIENCE,https://www.pulse.ng/news/world/an-irresistible-scent-makes-locusts-swarm-study-finds/jy784jw,pulse.ng,2020-08-12 15:14:19,"An irresistible scent makes locusts swarm, study finds",en,1
2,SCIENCE,https://www.express.co.uk/news/science/1322607/artificial-intelligence-warning-machine-learning-algorithm-...,express.co.uk,2020-08-13 21:01:00,Artificial intelligence warning: AI will know us better than we know ourselves,en,2
3,SCIENCE,https://www.ndtv.com/world-news/glaciers-could-have-sculpted-mars-valleys-study-2273648,ndtv.com,2020-08-03 22:18:26,Glaciers Could Have Sculpted Mars Valleys: Study,en,3
4,SCIENCE,https://www.thesun.ie/tech/5742187/perseid-meteor-shower-tonight-time-uk-see/,thesun.ie,2020-08-12 19:54:36,Perseid meteor shower 2020: What time and how to see the huge bright FIREBALLS over UK again tonight,en,4
...,...,...,...,...,...,...,...
108769,NATION,https://www.vanguardngr.com/2020/08/pdp-governors-forum-urges-security-agencies-to-re-strategise-to-curb-i...,vanguardngr.com,2020-08-08 02:40:00,PDP governors’ forum urges security agencies to re-strategise to curb insecurity,en,108769
108770,BUSINESS,https://www.patentlyapple.com/patently-apple/2020/08/in-q2-20-apple-dominated-the-premium-smartphone-segme...,patentlyapple.com,2020-08-08 01:27:12,"In Q2-20, Apple Dominated the Premium Smartphone Segment in India as they plan to be a bigger Player in th...",en,108770
108771,HEALTH,https://www.belfastlive.co.uk/news/health/coronavirus-northern-ireland-full-breakdown-18758815,belfastlive.co.uk,2020-08-12 17:01:00,Coronavirus Northern Ireland: Full breakdown shows 29 new positive cases in past 24 hours,en,108771
108772,ENTERTAINMENT,https://www.thenews.com.pk/latest/696364-paul-mccartney-details-post-beatles-distress-and-how-he-blames-hi...,thenews.com.pk,2020-08-05 04:59:00,Paul McCartney details post-Beatles distress and how he blames himself for the split,en,108772


In [4]:
print("Shape:", pdf.shape)
print("\nColumn types:"); print(pdf.dtypes)
print("\nMissing values per column:"); print(pdf.isna().sum())
print("\nArticles per topic:"); print(pdf["topic"].value_counts())
print("\nLanguages:", pdf["lang"].unique(), "| domains:", pdf["domain"].nunique())
print("Title length (words):", pdf["title"].str.split().str.len().describe().round(1).to_dict())

Shape: (108774, 7)

Column types:
topic               str
link                str
domain              str
published_date      str
title               str
lang                str
id                int64
dtype: object

Missing values per column:
topic             0
link              0
domain            0
published_date    0
title             0
lang              0
id                0
dtype: int64

Articles per topic:
topic
TECHNOLOGY       15000
HEALTH           15000
WORLD            15000
ENTERTAINMENT    15000
SPORTS           15000
BUSINESS         15000
NATION           15000
SCIENCE           3774
Name: count, dtype: int64

Languages: <ArrowStringArray>
['en']
Length: 1, dtype: str | domains: 5164


Title length (words): {'count': 108774.0, 'mean': 11.4, 'std': 3.6, 'min': 1.0, '25%': 9.0, '50%': 11.0, '75%': 13.0, 'max': 51.0}


**Observations.** 108 774 English news articles with 6 columns plus our `id`:
- `topic`: the label (8 classes, 15 000 articles each except SCIENCE with 3 774);
- `link`, `domain`: source;
- `published_date`: a string, not parsed;
- `title`: the text we will embed;
- `lang`: always `en`.

There are **no missing values**. Titles are short (about 11 words on average), which suits sentence-embedding models. The file is sorted by topic, so the first 1 000 rows contain only SCIENCE, TECHNOLOGY and HEALTH articles; we keep this subset as asked, which is fine for queries such as *animal* or *space*.

In [5]:
# Create a manageable subset (first 1000 rows)
pdf_subset = pdf.head(1000).copy()
print(pdf_subset["topic"].value_counts())
pdf_subset[["id", "title"]].head()

topic
TECHNOLOGY    587
HEALTH        232
SCIENCE       181
Name: count, dtype: int64


,id,title
0,0,A closer look at water-splitting's solar fuel potential
1,1,"An irresistible scent makes locusts swarm, study finds"
2,2,Artificial intelligence warning: AI will know us better than we know ourselves
3,3,Glaciers Could Have Sculpted Mars Valleys: Study
4,4,Perseid meteor shower 2020: What time and how to see the huge bright FIREBALLS over UK again tonight


## 🌟 Exercise 2 · Vectorization with Sentence Transformers

In [6]:
def example_create_fn(doc1: pd.Series) -> InputExample:
    """
    Helper function that outputs a sentence_transformer guid, label, and text.
    """
    return InputExample(guid=str(doc1.name), texts=[doc1["title"]], label=0.0)

# one InputExample per news title (guid = row index, label unused for inference)
faiss_train_examples = pdf_subset.apply(lambda x: example_create_fn(x), axis=1).tolist()
faiss_train_examples[:10]

In [7]:
model = SentenceTransformer("all-MiniLM-L6-v2")
print(model)

titles_list = pdf_subset["title"].tolist()
t0 = time.time()
faiss_title_embedding = model.encode(titles_list, convert_to_numpy=True, show_progress_bar=False, batch_size=64)
print(f"Encoded {len(titles_list)} titles in {time.time()-t0:.1f}s")
len(faiss_title_embedding), len(faiss_title_embedding[0])

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)


Encoded 1000 titles in 8.0s


(1000, 384)

We get **1 000 embeddings** (one per title), each of **384 dimensions**: `all-MiniLM-L6-v2` maps any sentence to a dense 384-d vector, where semantically close titles get close vectors.

## 🌟 Exercise 3 · FAISS indexing and search

In [8]:
pdf_to_index = pdf_subset
id_index = pdf_to_index["id"].to_numpy().astype(np.int64)

content_encoded_normalized = faiss_title_embedding.astype("float32").copy()
faiss.normalize_L2(content_encoded_normalized)          # unit-length vectors -> inner product = cosine similarity
print("norms after normalisation:", np.linalg.norm(content_encoded_normalized, axis=1)[:5].round(4))

index_content = faiss.IndexIDMap(faiss.IndexFlatIP(len(faiss_title_embedding[0])))
index_content.add_with_ids(content_encoded_normalized, id_index)
index_content.ntotal

norms after normalisation: [1. 1. 1. 1. 1.]


1000

In [9]:
def search_content(query, pdf_to_index, k=3):
    query_vector = model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(query_vector)

    # Perform the search
    top_k = index_content.search(query_vector, k)
    ids = top_k[1][0].tolist()
    similarities = top_k[0][0].tolist()

    # keep FAISS' ranking order (isin() alone would return the rows in DataFrame order)
    results = pdf_to_index.set_index("id").loc[ids].reset_index()
    results["similarities"] = similarities
    return results

display(search_content("animal", pdf_to_index, k=5)[["id", "topic", "title", "similarities"]])

,id,topic,title,similarities
0,176,TECHNOLOGY,Random: You Can Pick Up and Pet Cats in Assassin's Creed Valhalla,0.391902
1,975,HEALTH,Researchers explore social behavior of animals toward emerging infectious diseases,0.376784
2,99,TECHNOLOGY,Ghostwire: Tokyo confirms dog petting,0.344059
3,928,SCIENCE,Just Let This Lizard Be a Dinosaur,0.317387
4,762,SCIENCE,'Secret' life of sharks: Study reveals their surprising social networks,0.295497


In [10]:
for q in ["space exploration", "covid vaccine trial", "new smartphone release", "climate change"]:
    print(f"Query: {q!r}")
    display(search_content(q, pdf_to_index, k=3)[["topic", "title", "similarities"]])

Query: 'space exploration'


,topic,title,similarities
0,SCIENCE,The scramble for space at Earth’s outer limits,0.538844
1,SCIENCE,Orbital space tourism set for rebirth in 2021,0.510738
2,SCIENCE,Scientific research to widen on ISS as SpaceX eyes commercial crew missions,0.455425


Query: 'covid vaccine trial'


,topic,title,similarities
0,HEALTH,COVID-19 clinical trials lack diversity,0.657529
1,HEALTH,Human trials of coronavirus vaccine set to begin in Indonesia,0.652648
2,HEALTH,"BCG vaccine is safe, does not increase COVID risk: Researchers - Sentinelassam",0.619951


Query: 'new smartphone release'


,topic,title,similarities
0,TECHNOLOGY,"Motorola teases Sept 2020 smartphone launch, Digital News",0.682520
1,TECHNOLOGY,World's first smartphone with both 5G and an under-display camera will launch in September,0.604718
2,TECHNOLOGY,Oppo Reno 4 Pro hands-on: Flagship features on a midrange phone,0.585410


Query: 'climate change'


,topic,title,similarities
0,HEALTH,"If Climate Change Wasn't Bad Already, Rise in Temperature Could Unearth New Epidemics",0.603202
1,SCIENCE,"Climate change likely cause of woolly rhinoceros extinction, scientists say",0.489149
2,SCIENCE,Climate Scientists Sound the Alarm: Warming Greenland Ice Sheet Passes Point of No Return,0.443451


### Going further: exact KNN vs approximate (ANN) search on the full dataset
`IndexFlatIP` is an **exact** k-nearest-neighbour search: it compares the query with every vector, so its cost is O(N·d). On larger collections we use **approximate** indexes.
- **IVF** (inverted file): k-means clusters; only `nprobe` clusters are scanned.
- **HNSW**: a navigable small-world graph.

Below we embed all 108 774 titles and compare the indexes on **recall@10**, i.e. the share of the exact top-10 that the ANN index also returns, and on query latency.

In [11]:
t0 = time.time()
all_emb = model.encode(pdf["title"].tolist(), convert_to_numpy=True, batch_size=256, show_progress_bar=False).astype("float32")
faiss.normalize_L2(all_emb)
print(f"Embedded {len(all_emb)} titles in {time.time()-t0:.0f}s -> {all_emb.shape}, {all_emb.nbytes/1e6:.0f} MB")

d = all_emb.shape[1]
rng = np.random.default_rng(0)
query_ids = rng.choice(len(all_emb), 500, replace=False)
queries = all_emb[query_ids]

def bench(index, name, build_time):
    t0 = time.time(); _, I = index.search(queries, 11); ms = (time.time() - t0) / len(queries) * 1000
    return {"index": name, "build (s)": round(build_time, 1), "query (ms)": round(ms, 3), "I": I}

rows = []
t0 = time.time(); flat = faiss.IndexFlatIP(d); flat.add(all_emb); rows.append(bench(flat, "Flat (exact KNN)", time.time() - t0))
truth = rows[0]["I"]

t0 = time.time()
ivf = faiss.IndexIVFFlat(faiss.IndexFlatIP(d), d, 256, faiss.METRIC_INNER_PRODUCT)
ivf.train(all_emb); ivf.add(all_emb); bt = time.time() - t0
for nprobe in [1, 4, 16]:
    ivf.nprobe = nprobe; rows.append(bench(ivf, f"IVF256, nprobe={nprobe}", bt))

t0 = time.time()
hnsw = faiss.IndexHNSWFlat(d, 32, faiss.METRIC_INNER_PRODUCT); hnsw.add(all_emb); bt = time.time() - t0
for ef in [16, 64]:
    hnsw.hnsw.efSearch = ef; rows.append(bench(hnsw, f"HNSW32, efSearch={ef}", bt))

for r in rows:
    r["recall@10"] = np.mean([len(set(a[1:]) & set(b[1:])) / 10 for a, b in zip(r["I"], truth)])  # [1:] skips the query itself
ann = pd.DataFrame(rows).drop(columns="I")
ann

Embedded 108774 titles in 256s -> (108774, 384), 167 MB


,index,build (s),query (ms),recall@10
0,Flat (exact KNN),0.1,1.385,1.0000
1,"IVF256, nprobe=1",2.8,0.025,0.6144
2,"IVF256, nprobe=4",2.8,0.061,0.8310
3,"IVF256, nprobe=16",2.8,0.192,0.9468
4,"HNSW32, efSearch=16",5.6,0.022,0.9656
5,"HNSW32, efSearch=64",5.6,0.052,0.9828


## 🌟 Exercise 4 · ChromaDB collection and querying

In [12]:
chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))
collection_name = "my_news"

# If a collection with the same name exists, delete it to avoid conflicts
if any(c.name == collection_name for c in chroma_client.list_collections()):
    chroma_client.delete_collection(name=collection_name)

print(f"Creating collection: '{collection_name}'")
collection = chroma_client.create_collection(name=collection_name)

display(pdf_subset)

collection.add(
    documents=pdf_subset["title"][:100].tolist(),
    metadatas=[{"topic": topic} for topic in pdf_subset["topic"][:100].tolist()],
    ids=[f"id{x}" for x in pdf_subset["id"][:100].tolist()],
)
print("Documents in collection:", collection.count())

Creating collection: 'my_news'


,topic,link,domain,published_date,title,lang,id
0,SCIENCE,https://www.eurekalert.org/pub_releases/2020-08/dbnl-acl080620.php,eurekalert.org,2020-08-06 13:59:45,A closer look at water-splitting's solar fuel potential,en,0
1,SCIENCE,https://www.pulse.ng/news/world/an-irresistible-scent-makes-locusts-swarm-study-finds/jy784jw,pulse.ng,2020-08-12 15:14:19,"An irresistible scent makes locusts swarm, study finds",en,1
2,SCIENCE,https://www.express.co.uk/news/science/1322607/artificial-intelligence-warning-machine-learning-algorithm-...,express.co.uk,2020-08-13 21:01:00,Artificial intelligence warning: AI will know us better than we know ourselves,en,2
3,SCIENCE,https://www.ndtv.com/world-news/glaciers-could-have-sculpted-mars-valleys-study-2273648,ndtv.com,2020-08-03 22:18:26,Glaciers Could Have Sculpted Mars Valleys: Study,en,3
4,SCIENCE,https://www.thesun.ie/tech/5742187/perseid-meteor-shower-tonight-time-uk-see/,thesun.ie,2020-08-12 19:54:36,Perseid meteor shower 2020: What time and how to see the huge bright FIREBALLS over UK again tonight,en,4
...,...,...,...,...,...,...,...
995,TECHNOLOGY,https://www.androidcentral.com/mate-40-will-be-last-huawei-phone-kirin-processor,androidcentral.com,2020-08-07 17:12:33,The Mate 40 will be the last Huawei phone with a Kirin processor,en,995
996,SCIENCE,https://www.cnn.com/2020/08/17/africa/stone-age-grass-bedding-scn/index.html,cnn.com,2020-08-17 17:10:00,"Early humans knew how to make comfy, pest-free beds, study says",en,996
997,HEALTH,https://www.tenterfieldstar.com.au/story/6877652/regional-vic-set-for-virus-testing-blitz/,tenterfieldstar.com.au,2020-08-13 03:26:06,Regional Vic set for virus testing blitz,en,997
998,HEALTH,https://news.sky.com/story/coronavirus-trials-of-second-contact-tracing-app-launched-12048458,news.sky.com,2020-08-13 13:22:58,Coronavirus: Trials of second contact-tracing app launched,en,998


Documents in collection: 100


In [13]:
results = collection.query(query_texts=["space"], n_results=10)
print(json.dumps(results, indent=4))

{
    "ids": [
        [
            "id72",
            "id7",
            "id30",
            "id26",
            "id23",
            "id76",
            "id69",
            "id40",
            "id47",
            "id75"
        ]
    ],
    "embeddings": null,
    "documents": [
        [
            "Beck teams up with NASA and AI for 'Hyperspace' visual album experience",
            "Orbital space tourism set for rebirth in 2021",
            "NASA drops \"insensitive\" nicknames for cosmic objects",
            "\u2018It came alive:\u2019 NASA astronauts describe experiencing splashdown in SpaceX Dragon",
            "Hubble Uses Moon As \u201cMirror\u201d to Study Earth\u2019s Atmosphere \u2013 Proxy in Search of Potentially Habitable Planets Around Other Stars",
            "Australia's small yet crucial part in the mission to find life on Mars",
            "NASA Astronauts in SpaceX Capsule Splashdown in Gulf Of Mexico",
            "SpaceX's Starship spacecraft saw 150 mete

In [14]:
# Same results as a table (Chroma's default space is squared L2: smaller distance = more similar)
pd.DataFrame({"id": results["ids"][0], "distance": results["distances"][0],
              "topic": [m["topic"] for m in results["metadatas"][0]], "title": results["documents"][0]})

,id,distance,topic,title
0,id72,1.225036,TECHNOLOGY,Beck teams up with NASA and AI for 'Hyperspace' visual album experience
1,id7,1.308977,SCIENCE,Orbital space tourism set for rebirth in 2021
2,id30,1.391039,SCIENCE,"NASA drops ""insensitive"" nicknames for cosmic objects"
3,id26,1.406462,SCIENCE,‘It came alive:’ NASA astronauts describe experiencing splashdown in SpaceX Dragon
4,id23,1.439130,SCIENCE,Hubble Uses Moon As “Mirror” to Study Earth’s Atmosphere – Proxy in Search of Potentially Habitable Planet...
5,id76,1.489879,SCIENCE,Australia's small yet crucial part in the mission to find life on Mars
6,id69,1.572825,SCIENCE,NASA Astronauts in SpaceX Capsule Splashdown in Gulf Of Mexico
7,id40,1.573813,SCIENCE,SpaceX's Starship spacecraft saw 150 meters high
8,id47,1.583530,SCIENCE,NASA’s InSight lander shows what’s beneath Mars’ surface
9,id75,1.586463,SCIENCE,Alien base on Mercury: ET hunters claim to find huge UFO


Chroma embedded the 100 titles **automatically**, with its default embedding function (`all-MiniLM-L6-v2` run through ONNX). For *space* it ranks first the titles about NASA, astronauts, Mars, planets and the Moon. The returned JSON contains, for each query, the `ids`, `documents`, `metadatas` and `distances` of the 10 nearest neighbours.

Chroma also does what a plain library like FAISS does not do: it stores documents and **metadata**, and it can **filter** on that metadata. Below we restrict the search to a topic, and we create a second collection that uses cosine distance and reuses the embeddings we already computed.

In [15]:
print("Topics among the 100 docs:", pd.Series([m["topic"] for m in collection.get()["metadatas"]]).value_counts().to_dict())
filtered = collection.query(query_texts=["new discovery"], n_results=3, where={"topic": "TECHNOLOGY"})
print("\n'new discovery' restricted to TECHNOLOGY:")
for doc, dist in zip(filtered["documents"][0], filtered["distances"][0]):
    print(f"  {dist:.3f}  {doc}")

# bring-your-own embeddings + cosine distance on the full 1000-title subset
if any(c.name == "my_news_cosine" for c in chroma_client.list_collections()):
    chroma_client.delete_collection("my_news_cosine")
col_cos = chroma_client.create_collection("my_news_cosine", metadata={"hnsw:space": "cosine"})
col_cos.add(ids=[str(i) for i in pdf_subset["id"]], embeddings=faiss_title_embedding.tolist(),
            documents=pdf_subset["title"].tolist(), metadatas=[{"topic": t} for t in pdf_subset["topic"]])
res_cos = col_cos.query(query_embeddings=model.encode(["space"]).tolist(), n_results=5)
print("\n'space' on 1000 titles (cosine distance = 1 - cosine similarity):")
for doc, dist in zip(res_cos["documents"][0], res_cos["distances"][0]):
    print(f"  {dist:.3f}  {doc}")

Topics among the 100 docs: {'SCIENCE': 53, 'TECHNOLOGY': 47}



'new discovery' restricted to TECHNOLOGY:
  1.523  Alexa helps players navigate sci-fi world in first 3 episodes of Amazon audio adventure ‘Starfinder’
  1.555  Beck teams up with NASA and AI for 'Hyperspace' visual album experience
  1.568  Ghostwire: Tokyo confirms dog petting



'space' on 1000 titles (cosine distance = 1 - cosine similarity):
  0.444  The scramble for space at Earth’s outer limits
  0.613  Beck teams up with NASA and AI for 'Hyperspace' visual album experience
  0.624  Outrage after NASA ‘goes woke’ and renames ‘insensitive’ space objects
  0.645  NASA astronauts "This is an extraordinary day to be in space ..." shoot music videos in the orbit
  0.646  Land of a billion faces


## 🌟 Exercise 5 · Question answering with a Hugging Face model

The exercise asks for a **causal LM** (`AutoModelForCausalLM`, e.g. GPT-2), while the template suggests `google/flan-t5-small`. We use both:
- **`Qwen/Qwen2.5-0.5B-Instruct`**, a small (0.5 B parameters) instruction-tuned causal LM. It runs on CPU, and unlike GPT-2 it was trained to follow instructions, so it can actually *answer* a question from a context. GPT-2 only continues text.
- **`google/flan-t5-small`**, the template's seq2seq alternative, for comparison.

In [16]:
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
lm_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32)

pipe = pipeline(
    "text-generation",
    model=lm_model,
    tokenizer=tokenizer,
    max_new_tokens=512,   # maximum number of tokens to generate
    device_map="auto",    # GPU if available, else CPU
)
print(f"{model_id}: {sum(p.numel() for p in lm_model.parameters())/1e6:.0f}M parameters on {lm_model.device}")

Qwen/Qwen2.5-0.5B-Instruct: 494M parameters on cpu


In [17]:
question = "What's the latest news on space development?"
context = " ".join([f"#{str(i)}" for i in results["documents"][0]])     # exercise template (as given)
print("Context as built by the template:", context[:200], "...")

Context as built by the template: #Beck teams up with NASA and AI for 'Hyperspace' visual album experience #Orbital space tourism set for rebirth in 2021 #NASA drops "insensitive" nicknames for cosmic objects #‘It came alive:’ NASA as ...


Note: the template's `f"#{str(i)}"` **replaces** each document with `#` followed by the document itself. It is easier for the model if each document is numbered and placed on its own line, so we build the context with `f"#{n} {doc}"` instead.

In [18]:
context = "\n".join(f"#{n} {doc}" for n, doc in enumerate(results["documents"][0], 1))
prompt_template = f"Relevant context: {context}\n\n The user's question: {question}"
print(prompt_template)

Relevant context: #1 Beck teams up with NASA and AI for 'Hyperspace' visual album experience
#2 Orbital space tourism set for rebirth in 2021
#3 NASA drops "insensitive" nicknames for cosmic objects
#4 ‘It came alive:’ NASA astronauts describe experiencing splashdown in SpaceX Dragon
#5 Hubble Uses Moon As “Mirror” to Study Earth’s Atmosphere – Proxy in Search of Potentially Habitable Planets Around Other Stars
#6 Australia's small yet crucial part in the mission to find life on Mars
#7 NASA Astronauts in SpaceX Capsule Splashdown in Gulf Of Mexico
#8 SpaceX's Starship spacecraft saw 150 meters high
#9 NASA’s InSight lander shows what’s beneath Mars’ surface
#10 Alien base on Mercury: ET hunters claim to find huge UFO

 The user's question: What's the latest news on space development?


In [19]:
def generate(prompt, max_new_tokens=200):
    # chat template = the format the instruct model was trained on; greedy decoding for reproducibility
    messages = [{"role": "system", "content": "You are a helpful assistant. Answer using only the provided news headlines. "
                                              "If the context does not contain the answer, say so."},
                {"role": "user", "content": prompt}]
    out = pipe(messages, max_new_tokens=max_new_tokens, do_sample=False)
    return out[0]["generated_text"][-1]["content"]

t0 = time.time()
lm_response = pipe([{"role": "user", "content": prompt_template}], do_sample=False)
print(lm_response[0]["generated_text"][-1]["content"])
print(f"\n[{time.time()-t0:.1f}s]")

The latest developments in space technology and exploration include:

- **SpaceX's Falcon Heavy rocket**: This is expected to launch more rockets into orbit, potentially increasing the number of missions that can be conducted from Earth.

- **NASA's Artemis program**: This includes plans to send humans back to the moon and eventually establish a permanent human presence there.

- **SpaceX's Crew Dragon capsule**: It has been used to transport crew members to and from the International Space Station (ISS), demonstrating its capabilities as a reusable spacecraft.

- **NASA's Mars rovers**: The Perseverance rover landed on Mars in February 2021, marking another significant milestone in the search for signs of past or present life on the red planet.

- **SpaceX's Starship**: This is designed to carry humans to Mars and beyond, showcasing advancements in both propulsion systems and manufacturing processes.

- **China's Tianhe supercluster**: China is also developing advanced space technolog

### 6. Experiment with different prompts and context windows

In [20]:
def rag_answer(question, k=3, collection=col_cos, show_context=True):
    hits = collection.query(query_embeddings=model.encode([question]).tolist(), n_results=k)
    ctx = "\n".join(f"#{n} {doc}" for n, doc in enumerate(hits["documents"][0], 1))
    if show_context:
        print(f"--- context (k={k}) ---\n{ctx}")
    prompt = f"Relevant context:\n{ctx}\n\nThe user's question: {question}\nAnswer in 2-3 sentences, citing the headline numbers."
    return generate(prompt)

q = "What's the latest news on space development?"
for k in [1, 3, 10]:
    t0 = time.time()
    print(f"\n===== k = {k} =====")
    print("ANSWER:", rag_answer(q, k=k), f"\n[{time.time()-t0:.1f}s]")


===== k = 1 =====
--- context (k=1) ---
#1 Outrage after NASA ‘goes woke’ and renames ‘insensitive’ space objects


ANSWER: NASA has gone "woke" and renamed some of its "insensitive" objects to more respectful terms. 
[1.7s]

===== k = 3 =====
--- context (k=3) ---
#1 Outrage after NASA ‘goes woke’ and renames ‘insensitive’ space objects
#2 Orbital space tourism set for rebirth in 2021
#3 Tonight offers best chance of spotting space station in night sky


ANSWER: NASA has gone "woke" and renamed some of its "insensitive" objects as part of efforts to improve public perception and understanding of space exploration. 
[2.4s]

===== k = 10 =====
--- context (k=10) ---
#1 Outrage after NASA ‘goes woke’ and renames ‘insensitive’ space objects
#2 Orbital space tourism set for rebirth in 2021
#3 Tonight offers best chance of spotting space station in night sky
#4 Justice Department opens probe into ex-NASA official, Boeing over space contract: sources
#5 Next PS5 Announcement 'Tentatively Scheduled' for This Month, Says New Report
#6 The scramble for space at Earth’s outer limits
#7 Microsoft news recap: Surface Duo to get 3 years of Android updates, Xbox Series X confirmed to launch in November, and more
#8 Scientific research to widen on ISS as SpaceX eyes commercial crew missions
#9 NASA astronauts "This is an extraordinary day to be in space ..." shoot music videos in the orbit
#10 Motorola teases Sept 2020 smartphone launch, Digital News


ANSWER: NASA has gone "woke" and renamed some space objects, while the next PS5 announcement is tentatively scheduled for this month. 
[2.7s]


In [21]:
# Without retrieval: the model can only use what it memorised during pre-training
print("NO CONTEXT:", generate(q + " Answer in 2-3 sentences.", max_new_tokens=150))

NO CONTEXT: NASA has announced plans to launch the Perseverance rover on its first Mars mission. The rover will be sent to the Red Planet by 2024 and will explore the planet's surface for signs of past life. NASA also announced that it is developing a new spacecraft called Orion, which could become the first crewed spacecraft to land on the Moon. The company is working on a plan to send humans to the Moon by 2024.


In [22]:
for q in ["What did researchers find about COVID-19?",
          "Are there any news about animals?",
          "Who won the football match yesterday?"]:     # not covered by the SCIENCE/TECH/HEALTH subset
    print(f"\n===== {q} =====")
    print("ANSWER:", rag_answer(q, k=5))


===== What did researchers find about COVID-19? =====
--- context (k=5) ---
#1 Scientists Say They've Decoded The Likely Order Of COVID-19 Symptoms
#2 Death, 12 More Cases Of COVID-19
#3 One more Covid-19 related death and 35 new cases reported
#4 COVID-19 update for Aug. 17: Here's the latest on coronavirus in B.C.
#5 Gargling with mouthwashes can lower spread of COVID-19, say scientists


ANSWER: Researchers have decoded the likely order of symptoms associated with the novel coronavirus (COVID-19).

===== Are there any news about animals? =====
--- context (k=5) ---
#1 Researchers explore social behavior of animals toward emerging infectious diseases
#2 We found the first cat in the UK with Covid-19 - but there's no need to panic
#3 Herbivores Are More Vulnerable To Extinct Then Predators, Research Study
#4 Ghostwire: Tokyo confirms dog petting
#5 'Secret' life of sharks: Study reveals their surprising social networks


ANSWER: Yes, there is an article on the topic of animal behavior and disease transmission.

===== Who won the football match yesterday? =====
--- context (k=5) ---
#1 Here’s How The San Francisco Shock Won The Overwatch League Countdown Cup
#2 Game review – Fall Guys: Ultimate Knockout
#3 EA Sports UFC 4 review: Split decision
#4 FIFA 21 drops two goal celebrations to curb player toxicity
#5 PUBG MOBILE Celebrates Pakistan's 74th Independence Day -


ANSWER: The user's question asks about who won the football match yesterday. According to the given headlines, the winner of the football match was #1, which is "Here’s How The San Francisco Shock Won The Overwatch League Countdown Cup".


In [23]:
# The template's alternative: a small seq2seq model (flan-t5-small, 77M parameters)
t5_pipe = pipeline("text2text-generation", model="google/flan-t5-small", max_new_tokens=64)
context_docs = results["documents"][0][:3]
context = " ".join(context_docs)
prompt = f"Answer the question using only the context.\nContext: {context}\nQuestion: {question}\nAnswer:\n"
response = t5_pipe(prompt)[0]["generated_text"]
print(response)

NASA drops "insensitive" nicknames for cosmic objects


## Analysis and conclusions

**Retrieval (FAISS and Chroma) works well.**
- `covid vaccine trial`, `new smartphone release` and `space exploration` return clearly relevant titles, with cosine similarities of 0.5–0.7.
- `animal` is more ambiguous: the top hit is *"You Can Pick Up and Pet Cats in Assassin's Creed Valhalla"* (0.39). The embedding captures the *animal* meaning (cats, dogs, lizards, sharks) regardless of topic. This is semantic rather than keyword search, with its strengths and its surprises.
- In Chroma, the *space* query on the 100 documents ranks NASA/SpaceX/Mars titles. The first hit, *"Beck… 'Hyperspace' visual album"*, shows that a single shared word ("space", "NASA") can dominate a very short title.

**KNN vs ANN (all 108 774 titles).**
- The exact `IndexFlatIP` takes about 1.4 ms per query and is already fast at this scale.
- **HNSW** reaches **96.6 % recall@10 at 0.022 ms** (60× faster), and 98.3 % with `efSearch=64`.
- **IVF** needs `nprobe=16` to reach 94.7 %; with `nprobe=1` it falls to 61 %.
- ANN indexes trade a little recall for a large speed-up, and their parameters (`nprobe`, `efSearch`) move that trade-off. Below about 100 k vectors, an exact flat index is often good enough.

**Library vs database.**
- FAISS is a *library*: we embed the texts ourselves, manage the IDs (`IndexIDMap`) and look up the rows in pandas ourselves.
- Chroma is a *vector database*. It stores the documents together with their metadata, embeds texts automatically, supports metadata filters (`where={"topic": ...}`), lets us choose the distance (L2 by default, cosine with `hnsw:space`), and accepts our own embeddings.

**Generation: what we observed.**
1. **The template prompt without instructions led to hallucination.** With the 10 retrieved headlines, Qwen wrote a generic answer from its own memory instead of the context. It mentioned Falcon Heavy, Artemis and *Perseverance "landed in February 2021"* (after the August 2020 news it was given), plus a non-existent "Tianhe supercluster / Tiangong-5". Almost none of it comes from the context.
2. **A grounding instruction fixes this.** A system message ("answer using only the provided headlines…") plus a chat-formatted prompt makes the answers short and faithful to the retrieved titles, e.g. *"NASA has gone 'woke' and renamed some space objects…"*.
3. **The context size matters.**
   - With k=1 the answer is correct but narrow.
   - k=3 adds little.
   - With k=10 irrelevant headlines get in (PS5, Microsoft, Motorola), and the model mixes them into the answer (*"…while the next PS5 announcement is tentatively scheduled"*).

   More context is not always better: precision of retrieval matters as much as recall.
4. **Without retrieval** the model invents facts (*"Perseverance… will be sent to the Red Planet by 2024"*). This is exactly the problem RAG addresses.
5. **Questions outside the corpus.** The 1 000-row subset contains no SPORTS articles, so *"Who won the football match yesterday?"* retrieves video-game headlines, and the model claims the answer is the Overwatch League. A production RAG needs a **similarity threshold** (refuse when the best score is low) and a stronger model to say "I don't know".
6. **`flan-t5-small` (77 M parameters)** just copies one headline. It is extractive and safe, but not a real answer.

**Best practices learnt:**
- normalise the vectors for cosine search;
- keep IDs and metadata next to the vectors;
- choose exact vs ANN according to collection size and latency needs;
- give the LLM a clear grounding instruction with numbered sources;
- tune k;
- filter low-similarity hits;
- prefer instruction-tuned models (GPT-2 would only continue the prompt).